In [16]:
 %run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_00_Pre_Pilot/Jupyter Notebooks/01_spark_adls_connectors.ipynb"

spark v3.5.7 connected .... ✅
ADLS Gen2 storage account 'clientdatastorage' Connected ....✅ 


In [17]:
import os, json, pathlib

KAGGLE_JSON = {
    "username": "rammopati",
    "key": "KGAT_de3a0d984bf7452f80edf4f21812df13"
}


kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_path = kaggle_dir / "kaggle.json"
with open(kaggle_path, "w") as f:
    json.dump(KAGGLE_JSON, f)

os.chmod(kaggle_path, 0o600)
print("✅ Kaggle token written to", kaggle_path)

✅ Kaggle token written to /Users/manojrammopati/.kaggle/kaggle.json


In [18]:
import os, shutil, subprocess, glob

download_dir = "./tmp/kaggle_bupa"

if os.path.exists(download_dir):
    shutil.rmtree(download_dir)
os.makedirs(download_dir, exist_ok=True)

def kaggle_download(slug, target):
    cmd = ["kaggle", "datasets", "download", "-d", slug, "-p", target, "--unzip"]
    print("Running:", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout or out.stderr)

kaggle_download("rohitrox/healthcare-provider-fraud-detection-analysis", download_dir)
kaggle_download("anmolkumar/health-insurance-cross-sell-prediction", download_dir)

glob.glob(download_dir + "/*")[:15]


Running: kaggle datasets download -d rohitrox/healthcare-provider-fraud-detection-analysis -p ./tmp/kaggle_bupa --unzip
Dataset URL: https://www.kaggle.com/datasets/rohitrox/healthcare-provider-fraud-detection-analysis
License(s): CC0-1.0


Running: kaggle datasets download -d anmolkumar/health-insurance-cross-sell-prediction -p ./tmp/kaggle_bupa --unzip
Dataset URL: https://www.kaggle.com/datasets/anmolkumar/health-insurance-cross-sell-prediction
License(s): GPL-2.0




['./tmp/kaggle_bupa/Train_Inpatientdata-1542865627584.csv',
 './tmp/kaggle_bupa/Train_Outpatientdata-1542865627584.csv',
 './tmp/kaggle_bupa/Test_Inpatientdata-1542969243754.csv',
 './tmp/kaggle_bupa/Test_Beneficiarydata-1542969243754.csv',
 './tmp/kaggle_bupa/Train-1542865627584.csv',
 './tmp/kaggle_bupa/test.csv',
 './tmp/kaggle_bupa/Train_Beneficiarydata-1542865627584.csv',
 './tmp/kaggle_bupa/Test_Outpatientdata-1542969243754.csv',
 './tmp/kaggle_bupa/train.csv',
 './tmp/kaggle_bupa/sample_submission.csv',
 './tmp/kaggle_bupa/Test-1542969243754.csv']

In [19]:
from pyspark.sql import functions as F
import glob, os

base = download_dir

def try_read(patterns):
    for pat in patterns:
        matches = glob.glob(os.path.join(base, pat))
        if matches:
            return spark.read.option("header", True).csv(matches[0])
    return None

# Provider fraud dataset
inpatient   = try_read(["*Inpatientdata*.csv"])
outpatient  = try_read(["*Outpatientdata*.csv"])
beneficiary = try_read(["*Beneficiarydata*.csv"])
potential   = try_read(["*PotentialFraud*.csv", "*Provider*.csv", "*Train-*.csv"])

# Cross-sell dataset (for policies + members)
crosssell   = try_read(["train.csv", "Train.csv", "*health_insurance*train*.csv"])

print("Loaded:",
      "inpatient", inpatient.count() if inpatient else None,
      "outpatient", outpatient.count() if outpatient else None,
      "beneficiary", beneficiary.count() if beneficiary else None,
      "potential", potential.count() if potential else None,
      "crosssell", crosssell.count() if crosssell else None)


Loaded: inpatient 40474 outpatient 517737 beneficiary 63968 potential 5410 crosssell 381109


In [20]:
prov_id_col = "Provider"
if potential and prov_id_col not in potential.columns:
    for c in potential.columns:
        if c.lower() in ("provider","provider_id","providerid"):
            prov_id_col = c
            break

providers_raw = potential.select(
    F.col(prov_id_col).alias("Provider_ID"),
    *[F.col(c) for c in potential.columns if c.lower() != prov_id_col.lower()]
)

fraud_col = None
for c in providers_raw.columns:
    if c.lower() in ("potentialfraud","fraud_flag","fraudlabel","is_fraud"):
        fraud_col = c
        break

if fraud_col:
    providers_bronze = providers_raw.select(
        F.col("Provider_ID"),
        F.when(F.lower(F.col(fraud_col)).isin("yes","y","true","1"), 1)
         .when(F.expr(f"try_cast({fraud_col} as int)") == 1, 1)
         .otherwise(0).alias("Fraud_Flag")
    ).dropDuplicates(["Provider_ID"])
else:
    providers_bronze = providers_raw.select("Provider_ID").dropDuplicates()\
                                    .withColumn("Fraud_Flag", F.lit(0))

providers_bronze.show(5, truncate=False)


+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|PRV51262   |0         |
|PRV51606   |0         |
|PRV51657   |0         |
|PRV51665   |0         |
|PRV51749   |0         |
+-----------+----------+
only showing top 5 rows



In [21]:
def unify_claims(df):
    if df is None:
        return None
    c = {col.lower(): col for col in df.columns}
    return df.select(
        F.col(c.get("claimid","ClaimID")).alias("Claim_ID"),
        F.col(c.get("provider","Provider")).alias("Provider_ID"),
        F.col(c.get("beneid","BeneID")).alias("Member_Key"),
        F.to_date(F.col(c.get("claimstartdt","ClaimStartDt")), "MM/dd/yyyy").alias("Date_Reported"),
        F.to_date(F.col(c.get("claimenddt","ClaimEndDt")), "MM/dd/yyyy").alias("Date_Settled"),
        F.col(c.get("inscclaimamtreimbursed","InscClaimAmtReimbursed")).cast("double").alias("Payout_GBP")
    )

clm_in  = unify_claims(inpatient)
clm_out = unify_claims(outpatient)

if clm_in and clm_out:
    claims_union = clm_in.unionByName(clm_out, allowMissingColumns=True)
elif clm_in:
    claims_union = clm_in
elif clm_out:
    claims_union = clm_out
else:
    raise Exception("Missing inpatient/outpatient CSVs.")

claims_bronze = (claims_union
  .withColumn("Claim_Amount_GBP",
              (F.col("Payout_GBP") * F.rand(seed=7) * 0.6 + F.col("Payout_GBP")).cast("double"))
  .withColumn("Fraud_Label", (F.rand(seed=9) < F.lit(0.015)).cast("int"))
  .withColumn("Claim_Type", F.when(F.rand(seed=11) < 0.45,"Hospital")
                            .when(F.rand(seed=12) < 0.65,"Outpatient")
                            .when(F.rand(seed=13) < 0.72,"Dental")
                            .when(F.rand(seed=14) < 0.77,"Maternity")
                            .when(F.rand(seed=15) < 0.80,"Travel")
                            .otherwise("Physio"))
  .withColumn("Claim_Status", F.when(F.rand(seed=21) < 0.70,"Settled")
                               .when(F.rand(seed=22) < 0.88,"Pending")
                               .when(F.rand(seed=23) < 0.98,"Rejected")
                               .otherwise("Withdrawn"))
  .withColumn("Provider_ID", F.when(F.rand(seed=31) < 0.07, F.lit(None)).otherwise(F.col("Provider_ID")))
  .withColumn("Claim_Type", F.when(F.rand(seed=32) < 0.07, F.lit(None)).otherwise(F.col("Claim_Type")))
)

claims_bronze.show(5)


+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|Claim_ID|Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|  Claim_Amount_GBP|Fraud_Label|Claim_Type|Claim_Status|
+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|CLM46614|   PRV55912| BENE11001|         NULL|        NULL|   26000.0|30397.346951642197|          0|  Hospital|     Settled|
|CLM66048|   PRV55907| BENE11001|         NULL|        NULL|    5000.0|  6379.96457105169|          0|  Hospital|     Settled|
|CLM68358|   PRV56046| BENE11001|         NULL|        NULL|    5000.0| 7307.291499809586|          0|  Hospital|     Settled|
|CLM38412|   PRV52405| BENE11011|         NULL|        NULL|    5000.0| 6989.424384581965|          0|  Hospital|     Settled|
|CLM63689|   PRV56614| BENE11014|         NULL|        NULL|   10000.0|13123.302569965643|          0|  Hospita

In [22]:
c = {col.lower(): col for col in crosssell.columns}
age_col       = c.get("age","Age")
prem_col      = c.get("annual_premium","Annual_Premium")
region_col    = c.get("region_code","Region_Code")
channel_col   = c.get("policy_sales_channel","Policy_Sales_Channel")
gender_col    = c.get("gender","Gender")
veh_damage_col= c.get("vehicle_damage","Vehicle_Damage")
id_col        = c.get("id","id")

def try_double(expr_str):
    return F.expr(f"""
      try_cast(
        regexp_replace(
          regexp_replace({expr_str}, '[,\\s]', ''),
          '[^0-9\\.]', ''
        )
      as double)
    """)

def try_int(expr_str):
    return F.expr(f"""
      try_cast(
        regexp_replace({expr_str}, '[^0-9]', '')
      as int)
    """)

crosssell_clean = (
    crosssell
      .withColumn("Age_double", try_double(f"`{age_col}`"))
      .withColumn("Age_int",
                  F.when(F.col("Age_double").isNull(), F.lit(None).cast("int"))
                   .otherwise(F.floor(F.col("Age_double")).cast("int")))
      .withColumn("Age_int",
                  F.when((F.col("Age_int") >= 0) & (F.col("Age_int") <= 110), F.col("Age_int"))
                   .otherwise(F.lit(None).cast("int")))
      .withColumn("Annual_Premium_double", try_double(f"`{prem_col}`"))
      .withColumn("Region_code_str",
                  F.expr(f"regexp_replace(`{region_col}`, '[^0-9]', '')"))
      .withColumn("Region_code_str",
                  F.when(F.length(F.col("Region_code_str"))==0, None)
                   .otherwise(F.col("Region_code_str")))
      .withColumn("Policy_Sales_Channel_int", try_int(f"`{channel_col}`"))
)

policy_base = (crosssell_clean
  .withColumn("Policy_ID",  F.concat(F.lit("P_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Customer_ID",F.concat(F.lit("CUST_"), F.lpad(F.col(id_col).cast("string"), 7, "0")))
  .withColumn("Annual_Premium_GBP", F.col("Annual_Premium_double"))
  .withColumn("Product_Line",
              F.when(F.rand(1) < 0.60,"Health")
               .when(F.rand(2) < 0.70,"Dental")
               .when(F.rand(3) < 0.85,"Motor")
               .when(F.rand(4) < 0.95,"Accident").otherwise("Travel"))
  .withColumn("Channel",
              F.when(F.col("Policy_Sales_Channel_int") < 20,"Online")
               .when(F.col("Policy_Sales_Channel_int") < 40,"Agent")
               .when(F.col("Policy_Sales_Channel_int") < 70,"Broker")
               .otherwise("Partner"))
  .withColumn("Policy_Start_Date", F.expr("date_sub(current_date(), cast(rand(5)*365*5 as int))"))
  .withColumn("Policy_End_Date",   F.expr("date_add(Policy_Start_Date, cast(180 + rand(6)*360 as int))"))
  .withColumn("Sum_Insured_GBP", (F.col("Annual_Premium_GBP")/F.lit(0.02) * (1+F.rand(7)*0.3)).cast("double"))
  .withColumn("Renewal_Offered_Flag", (F.rand(8) < 0.75).cast("int"))
  .withColumn("Renewal_Accepted_Flag", (F.rand(9) < 0.70).cast("int"))
  .withColumn("Discount_Offered_%", F.when(F.col("Renewal_Offered_Flag")==1, (F.rand(10)*20).cast("double")).otherwise(F.lit(0.0)))
  .select("Policy_ID","Customer_ID","Product_Line","Sum_Insured_GBP","Annual_Premium_GBP",
          "Policy_Start_Date","Policy_End_Date","Renewal_Offered_Flag","Renewal_Accepted_Flag",
          "Discount_Offered_%","Channel")
)

members_base = (crosssell_clean
  .withColumn("Member_ID", F.concat(F.lit("MEM_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Policy_ID", F.concat(F.lit("P_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Age", F.col("Age_int"))
  .withColumn("Gender", F.col(gender_col))
  .withColumn("BMI", (F.lit(27.5) + (F.rand(1)-0.5)*12).cast("double"))
  .withColumn("Smoker", F.when(F.col(veh_damage_col)=="Yes","Y").otherwise("N"))
  .withColumn("Chronic_Disease", F.when(F.rand(2)<0.12,"Diabetes")
                                  .when(F.rand(3)<0.27,"Hypertension")
                                  .when(F.rand(4)<0.35,"Asthma")
                                  .otherwise("None"))
  .withColumn("Employment_Status", F.when(F.rand(5)<0.55,"Employed")
                                     .when(F.rand(6)<0.67,"Retired")
                                     .when(F.rand(7)<0.75,"Student")
                                     .otherwise("Self-Employed"))
  .withColumn("Region", F.col("Region_code_str"))
  .select("Member_ID","Policy_ID","Age","Gender","BMI","Smoker","Chronic_Disease","Employment_Status","Region")
)

policy_base.show(5)
members_base.show(5)


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
| Policy_ID| Customer_ID|Product_Line|   Sum_Insured_GBP|Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|P_00000001|CUST_0000001|      Dental| 2193748.339982436|           40454.0|       2025-11-05|     2026-05-29|                   1|                    0| 3.418994275911136|Partner|
|P_00000002|CUST_0000002|      Health|1908192.4592739474|           33536.0|       2025-06-28|     2026-09-16|                   1|                    1| 16.10228791601092|Partner|
|P_00000003|CUST_0000003|      Health|2356477.1034685415|           38294.0|       2022-02-11| 

In [23]:
policy_ids_df = policy_base.select("Policy_ID").withColumn("hash_key", F.rand(seed=42))
policy_ids_df = policy_ids_df.orderBy("hash_key").withColumn("rownum", F.monotonically_increasing_id())

claims_keyed = claims_bronze.withColumn("rownum", F.monotonically_increasing_id())

pol_count = policy_ids_df.count()

claims_keyed = claims_keyed.withColumn("join_key", (F.col("rownum") % F.lit(pol_count)))
policy_ids_df = policy_ids_df.withColumn("join_key", (F.col("rownum") % F.lit(pol_count)))\
                             .select("Policy_ID","join_key")

claims_final = (claims_keyed.join(policy_ids_df, "join_key", "left")
                             .drop("rownum","hash_key","join_key"))

claims_final.show(5)


+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+----------+
|Claim_ID|Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|  Claim_Amount_GBP|Fraud_Label|Claim_Type|Claim_Status| Policy_ID|
+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+----------+
|CLM46614|   PRV55912| BENE11001|         NULL|        NULL|   26000.0|30397.346951642197|          0|  Hospital|     Settled|P_00294224|
|CLM46614|   PRV55912| BENE11001|         NULL|        NULL|   26000.0|30397.346951642197|          0|  Hospital|     Settled|P_00287006|
|CLM66048|   PRV55907| BENE11001|         NULL|        NULL|    5000.0|  6379.96457105169|          0|  Hospital|     Settled|P_00343741|
|CLM66048|   PRV55907| BENE11001|         NULL|        NULL|    5000.0|  6379.96457105169|          0|  Hospital|     Settled|P_00219645|
|CLM68358|   PRV56046| BENE11001| 

In [24]:
raw_container = "rawdata"
raw_base = f"abfss://{raw_container}@{storage_account1}.dfs.core.windows.net"

paths = {
    "policies":   f"{raw_base}/policies",
    "members":    f"{raw_base}/members",
    "claims":     f"{raw_base}/claims",
    "providers":  f"{raw_base}/providers",
}
paths


{'policies': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/policies',
 'members': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/members',
 'claims': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/claims',
 'providers': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers'}

In [25]:
from py4j.java_gateway import java_import

def rm_path(path: str, storage_account: str):
    sc = spark.sparkContext
    hconf = sc._jsc.hadoopConfiguration()

    # ---- FORCE OAUTH CONF INTO HADOOP CONF ----
    hconf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
    hconf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
              "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    hconf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
    hconf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
    hconf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
              f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

    java_import(sc._jvm, "java.net.URI")
    java_import(sc._jvm, "org.apache.hadoop.fs.Path")
    java_import(sc._jvm, "org.apache.hadoop.fs.FileSystem")

    uri = sc._jvm.URI(path)
    fs = sc._jvm.FileSystem.get(uri, hconf)   # now it sees OAuth
    p = sc._jvm.Path(path)

    if fs.exists(p):
        fs.delete(p, True)
        print("🧹 Deleted:", path)
    else:
        print("✅ Path not found:", path)


In [26]:
providers_small = providers_bronze.coalesce(1)
policies_small  = policy_base.coalesce(1)
members_small   = members_base.coalesce(1)

# use claims_bronze if you're doing enterprise-correct version
claims_small    = claims_bronze.coalesce(2)

for k, v in paths.items():
    rm_path(v, storage_account1)

providers_small.write.format("delta").mode("overwrite").save(paths["providers"])
claims_small.write.format("delta").mode("overwrite").save(paths["claims"])
members_small.write.format("delta").mode("overwrite").save(paths["members"])
policies_small.write.format("delta").mode("overwrite").save(paths["policies"])

print("✅ RAWDATA INGESTION DONE.")


✅ Path not found: abfss://rawdata@clientdatastorage.dfs.core.windows.net/policies
✅ Path not found: abfss://rawdata@clientdatastorage.dfs.core.windows.net/members
✅ Path not found: abfss://rawdata@clientdatastorage.dfs.core.windows.net/claims
✅ Path not found: abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers


✅ RAWDATA INGESTION DONE.


In [27]:
policies_df  = spark.read.format("delta").load(paths["policies"])
members_df   = spark.read.format("delta").load(paths["members"])
claims_df    = spark.read.format("delta").load(paths["claims"])
providers_df = spark.read.format("delta").load(paths["providers"])

print("Policies :", policies_df.count())
print("Members  :", members_df.count())
print("Claims   :", claims_df.count())
print("Providers:", providers_df.count())

policies_df.show(5, truncate=False)


Policies : 381109
Members  : 381109
Claims   : 558211
Providers: 5410


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|P_00000001|CUST_0000001|Dental      |2193748.339982436 |40454.0           |2025-11-05       |2026-05-29     |1                   |0                    |3.418994275911136 |Partner|
|P_00000002|CUST_0000002|Health      |1908192.4592739474|33536.0           |2025-06-28       |2026-09-16     |1                   |1                    |16.10228791601092 |Partner|
|P_00000003|CUST_0000003|Health      |2356477.1034685415|38294.0           |2022-02-11       |2

25/12/19 04:13:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1008005 ms exceeds timeout 120000 ms
25/12/19 04:13:55 WARN SparkContext: Killing executors is not supported by current scheduler.
25/12/19 04:14:02 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$